In [78]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


In [ ]:
cd "/Users/apple/AI Matics/Fibro/ROL Project"
uvicorn api.main:app --reload

cd "/Users/apple/AI Matics/Fibro/ROL Project/frontend"
npm run dev


In [1]:
file_path = "/Users/apple/AI Matics/Fibro/ROL Project/data/ROL Working.xlsx"

df = pd.read_excel(file_path, sheet_name="Data")

df.head()

NameError: name 'pd' is not defined

In [80]:
df = df[
    [
        "OA Date",
        "Item Code",
        "Sum of Sales_Qty"
    ]
].copy()

df.head()

,OA Date,Item Code,Sum of Sales_Qty
0,2025-01-02,4960.85.125.250,4
1,2025-01-02,4960.85.048.150,6
2,2025-01-02,4960.85.075.150,2
3,2025-01-02,4960.85.100.150,1
4,2025-01-02,4960.85.048.100,2


In [81]:
df["OA Date"] = pd.to_datetime(df["OA Date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4817 entries, 0 to 4816
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   OA Date           4817 non-null   datetime64[ns]
 1   Item Code         4817 non-null   object        
 2   Sum of Sales_Qty  4817 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 113.0+ KB


In [82]:
df["Year"] = df["OA Date"].dt.year
df["Week"] = df["OA Date"].dt.isocalendar().week

df.head()

,OA Date,Item Code,Sum of Sales_Qty,Year,Week
0,2025-01-02,4960.85.125.250,4,2025,1
1,2025-01-02,4960.85.048.150,6,2025,1
2,2025-01-02,4960.85.075.150,2,2025,1
3,2025-01-02,4960.85.100.150,1,2025,1
4,2025-01-02,4960.85.048.100,2,2025,1


In [83]:
weekly = (
    df.groupby(
        [
            "Item Code",
            "Year",
            "Week"
        ],
        as_index=False
    )["Sum of Sales_Qty"]
    .sum()
)

weekly.head()

,Item Code,Year,Week,Sum of Sales_Qty
0,4960.85.028.075,2025,6,2
1,4960.85.028.075,2025,23,2
2,4960.85.028.075,2025,29,48
3,4960.85.028.075,2025,31,4
4,4960.85.028.075,2025,32,2


In [84]:
summary = (
    weekly.groupby(
        "Item Code",
        as_index=False
    )["Sum of Sales_Qty"]
    .sum()
)

summary.rename(
    columns={
        "Sum of Sales_Qty": "Total Sales"
    },
    inplace=True
)

summary = summary.sort_values(
    by="Total Sales",
    ascending=False
).reset_index(drop=True)

summary.head()

,Item Code,Total Sales
0,4960.85.100.100,6500
1,4960.85.100.125,4118
2,4960.85.075.100,3320
3,4960.85.100.150,3191
4,4960.85.125.125,2244


In [85]:
grand_total = summary["Total Sales"].sum()

summary["Contribution"] = (
    summary["Total Sales"] / grand_total
).round(4)

summary.head()

,Item Code,Total Sales,Contribution
0,4960.85.100.100,6500,0.1920
1,4960.85.100.125,4118,0.1216
2,4960.85.075.100,3320,0.0981
3,4960.85.100.150,3191,0.0943
4,4960.85.125.125,2244,0.0663


In [86]:
summary["per Month"] = (
    summary["Total Sales"] / 18
).round().astype(int)

summary.head()

,Item Code,Total Sales,Contribution,per Month
0,4960.85.100.100,6500,0.1920,361
1,4960.85.100.125,4118,0.1216,229
2,4960.85.075.100,3320,0.0981,184
3,4960.85.100.150,3191,0.0943,177
4,4960.85.125.125,2244,0.0663,125


In [87]:
# def volume_logic(total_sales):
#     if total_sales <= 300:
#         return 0
#     elif total_sales <= 600:
#         return 12
#     else:
#         return 24

# summary["Volume Logic"] = summary["Total Sales"].apply(volume_logic)

# summary.head(15)


# What client suggested intead of using the above volumn logic, for each product look for sum of sales
# for example for following product the sum of sales are in 2,4 qty. he said take mode. if we take mode
#     then repeated number is 2. then my frequency will be
# 0
# 1-2
# 2-4
# 4-6

# like that


# OA Date	Item Code	Sum of Sales_Qty	Year	Week No	Week	Week-Year
# 01/09/2025	4960.85.028.100	2	2025	2025-2	Week 2	Week 2-2025
# 02/25/2025	4960.85.028.100	2	2025	2025-9	Week 9	Week 9-2025
# 07/28/2025	4960.85.028.100	4	2025	2025-31	Week 31	Week 31-2025
# 08/03/2025	4960.85.028.100	2	2025	2025-31	Week 32	Week 32-2025
# 09/19/2025	4960.85.028.100	4	2025	2025-38	Week 38	Week 38-2025
# 11/20/2025	4960.85.028.100	2	2025	2025-47	Week 47	Week 47-2025
# 12/27/2025	4960.85.028.100	4	2025	2025-52	Week 52	Week 52-2025
# 01/30/2026	4960.85.028.100	4	2026	2026-5	Week 5	Week 5-2026

In [88]:
# Calculate mode weekly sales quantity for each Item Code
mode_qty = (
    weekly.groupby("Item Code")["Sum of Sales_Qty"]
    .agg(lambda x: x.mode().iloc[0])
)

# Merge into summary
summary = summary.merge(
    mode_qty.rename("Mode_Qty"),
    on="Item Code",
    how="left"
)

# Dynamic Volume Logic
def volume_logic(total_sales, mode_qty):

    if pd.isna(mode_qty) or mode_qty <= 0:
        return "0"

    if total_sales == 0:
        return "0"

    lower = ((total_sales - 1) // mode_qty) * mode_qty
    upper = lower + mode_qty

    if lower == 0:
        return f"1-{int(upper)}"
    else:
        return f"{int(lower)}-{int(upper)}"

summary["Volume Logic"] = summary.apply(
    lambda row: volume_logic(row["Total Sales"], row["Mode_Qty"]),
    axis=1
)

summary.head(15)

,Item Code,Total Sales,Contribution,per Month,Mode_Qty,Volume Logic
0,4960.85.100.100,6500,0.1920,361,86,6450-6536
1,4960.85.100.125,4118,0.1216,229,34,4114-4148
2,4960.85.075.100,3320,0.0981,184,18,3312-3330
3,4960.85.100.150,3191,0.0943,177,32,3168-3200
4,4960.85.125.125,2244,0.0663,125,16,2240-2256
5,4960.85.125.150,1881,0.0556,104,2,1880-1882
6,4960.85.150.150,1730,0.0511,96,8,1728-1736
7,4960.85.075.125,1079,0.0319,60,4,1076-1080
8,4960.85.058.100,883,0.0261,49,2,882-884
9,4960.85.075.150,860,0.0254,48,2,858-860


In [89]:
# item = "4960.85.028.075"

# tmp = weekly.loc[
#     weekly["Item Code"] == item,
#     ["Year", "Week", "Sum of Sales_Qty"]
# ].copy()

# print("Mode:")
# print(tmp["Sum of Sales_Qty"].mode().tolist())

# print("\nValue Counts:")
# print(tmp["Sum of Sales_Qty"].value_counts().sort_index())

# print("\nWeekly Data:")
# display(tmp)

In [90]:
mode_qty

Item Code
4960.85.028.075       2
4960.85.028.100       2
4960.85.028.150       8
4960.85.038.050       2
4960.85.038.075       4
4960.85.038.100       2
4960.85.038.125       2
4960.85.038.150       2
4960.85.038.200       4
4960.85.048.075       6
4960.85.048.100       4
4960.85.048.125       4
4960.85.048.150       2
4960.85.048.200       2
4960.85.058.075       8
4960.85.058.100       2
4960.85.058.125       4
4960.85.058.150       4
4960.85.058.200       8
4960.85.075.075      12
4960.85.075.100      18
4960.85.075.100.1     2
4960.85.075.125       4
4960.85.075.150       2
4960.85.075.200       8
4960.85.075.250       4
4960.85.100.100      86
4960.85.100.100.1     4
4960.85.100.125      34
4960.85.100.125.1     2
4960.85.100.150      32
4960.85.100.150.1     2
4960.85.100.200       2
4960.85.100.200.1     2
4960.85.100.250       4
4960.85.100.250.1     2
4960.85.100.300       4
4960.85.125.125      16
4960.85.125.125.1    16
4960.85.125.150       2
4960.85.125.200       2
4960.8

In [91]:
# item_code = "4960.85.100.100"

# item_code = "4960.85.058.100"

item_code = "4960.85.150.150" 
# item_code = "4960.85.125.125"
# item_code = "4960.85.028.075"



item_weekly = (
    weekly[weekly["Item Code"] == item_code]
    .sort_values(["Year", "Week"])
    .reset_index(drop=True)
)


item_weekly


,Item Code,Year,Week,Sum of Sales_Qty
0,4960.85.150.150,2025,2,30
1,4960.85.150.150,2025,3,20
2,4960.85.150.150,2025,4,16
3,4960.85.150.150,2025,5,10
4,4960.85.150.150,2025,6,4
...,...,...,...,...
57,4960.85.150.150,2026,16,29
58,4960.85.150.150,2026,17,24
59,4960.85.150.150,2026,18,7
60,4960.85.150.150,2026,20,55


In [92]:
# bin_size = 24

mode_qty = item_weekly["Sum of Sales_Qty"].mode()

print("bins size:", mode_qty)

#If there are multiple modes, mode() returns all of them and iloc[0] simply picks the 
# smallest one. It is better to make that explicit:
if mode_qty.empty:

    bin_size = 1

else:

    bin_size = int(mode_qty.min())

    
max_qty = item_weekly["Sum of Sales_Qty"].max()

ranges = [(0, 0)]

start = 1
while start <= max_qty:
    end = start + bin_size - 1
    ranges.append((start, end))
    start = end + 1

ranges

bins size: 0    8
Name: Sum of Sales_Qty, dtype: int64


[(0, 0),
 (1, 8),
 (9, 16),
 (17, 24),
 (25, 32),
 (33, 40),
 (41, 48),
 (49, 56),
 (57, 64),
 (65, 72),
 (73, 80),
 (81, 88),
 (89, 96),
 (97, 104),
 (105, 112),
 (113, 120),
 (121, 128),
 (129, 136),
 (137, 144),
 (145, 152),
 (153, 160),
 (161, 168)]

In [93]:
frequency = []

for low, high in ranges:

    if low == 0:
        count = (item_weekly["Sum of Sales_Qty"] == 0).sum()
    else:
        count = item_weekly["Sum of Sales_Qty"].between(low, high).sum()

    frequency.append([low, high, count])

frequency_df = pd.DataFrame(
    frequency,
    columns=["Lower", "Upper", "Frequency"]
)

TOTAL_WEEKS = 74 + 1

# Sum of all frequencies except the first (0-0 bucket)

non_zero_frequency = frequency_df.loc[1:, "Frequency"].sum()

# Remaining weeks are assigned to the 0 bucket

frequency_df.loc[0, "Frequency"] = TOTAL_WEEKS - non_zero_frequency

frequency_df

,Lower,Upper,Frequency
0,0,0,13
1,1,8,13
2,9,16,11
3,17,24,11
4,25,32,12
5,33,40,4
6,41,48,3
7,49,56,4
8,57,64,1
9,65,72,0


In [94]:
TOTAL_WEEKS = 74

frequency_df["Contribution"] = (
    frequency_df["Frequency"] / TOTAL_WEEKS
)

frequency_df

,Lower,Upper,Frequency,Contribution
0,0,0,13,0.175676
1,1,8,13,0.175676
2,9,16,11,0.148649
3,17,24,11,0.148649
4,25,32,12,0.162162
5,33,40,4,0.054054
6,41,48,3,0.040541
7,49,56,4,0.054054
8,57,64,1,0.013514
9,65,72,0,0.000000


In [95]:
frequency_df["Cum Probability"] = (

    frequency_df["Contribution"].cumsum()

)

frequency_df

,Lower,Upper,Frequency,Contribution,Cum Probability
0,0,0,13,0.175676,0.175676
1,1,8,13,0.175676,0.351351
2,9,16,11,0.148649,0.500000
3,17,24,11,0.148649,0.648649
4,25,32,12,0.162162,0.810811
5,33,40,4,0.054054,0.864865
6,41,48,3,0.040541,0.905405
7,49,56,4,0.054054,0.959459
8,57,64,1,0.013514,0.972973
9,65,72,0,0.000000,0.972973


In [96]:
frequency_df["Mid Point"] = np.where(
    frequency_df["Lower"] == 0,
    0,
    (frequency_df["Lower"] + frequency_df["Upper"]) / 2
)

frequency_df

,Lower,Upper,Frequency,Contribution,Cum Probability,Mid Point
0,0,0,13,0.175676,0.175676,0.0
1,1,8,13,0.175676,0.351351,4.5
2,9,16,11,0.148649,0.500000,12.5
3,17,24,11,0.148649,0.648649,20.5
4,25,32,12,0.162162,0.810811,28.5
5,33,40,4,0.054054,0.864865,36.5
6,41,48,3,0.040541,0.905405,44.5
7,49,56,4,0.054054,0.959459,52.5
8,57,64,1,0.013514,0.972973,60.5
9,65,72,0,0.000000,0.972973,68.5


In [97]:
frequency_df["Weighted Sum"] = (
    frequency_df["Mid Point"] *
    frequency_df["Contribution"]
)

frequency_df.round(
    {
        "Contribution": 3,
        "Cum Probability": 3,
        "Weighted Sum": 2
    } 
)

frequency_df

,Lower,Upper,Frequency,Contribution,Cum Probability,Mid Point,Weighted Sum
0,0,0,13,0.175676,0.175676,0.0,0.000000
1,1,8,13,0.175676,0.351351,4.5,0.790541
2,9,16,11,0.148649,0.500000,12.5,1.858108
3,17,24,11,0.148649,0.648649,20.5,3.047297
4,25,32,12,0.162162,0.810811,28.5,4.621622
5,33,40,4,0.054054,0.864865,36.5,1.972973
6,41,48,3,0.040541,0.905405,44.5,1.804054
7,49,56,4,0.054054,0.959459,52.5,2.837838
8,57,64,1,0.013514,0.972973,60.5,0.817568
9,65,72,0,0.000000,0.972973,68.5,0.000000


In [98]:
d_avg_week = frequency_df["Weighted Sum"].sum()

print(f"D Avg / Week = {d_avg_week:.2f}")

d_avg_month = round(d_avg_week * 4)

print(f"D Avg / Month : {d_avg_month}")

D Avg / Week = 22.80
D Avg / Month : 91


In [99]:
# SERVICE_LEVEL = 0.85

# dmax_row = frequency_df[
#     frequency_df["Cum Probability"] >= SERVICE_LEVEL
# ].iloc[0]

# d_max_week = dmax_row["Upper"]

# print(f"D Max / Week : {d_max_week}")


# d_max_month = d_max_week * 4

# print(f"D Max / Month : {d_max_month}")

import math

SERVICE_LEVEL = 0.85

# Row just below the service level
below = frequency_df[
    frequency_df["Cum Probability"] < SERVICE_LEVEL
].iloc[-1]

# Row just above (or equal to) the service level
above = frequency_df[
    frequency_df["Cum Probability"] >= SERVICE_LEVEL
].iloc[0]

# Linear interpolation
fraction = (
    (SERVICE_LEVEL - below["Cum Probability"]) /
    (above["Cum Probability"] - below["Cum Probability"])
)

d_max_week = (
    below["Upper"] +
    fraction * (above["Upper"] - below["Upper"])
)

# Standard rounding to nearest integer (0.5 rounds up)
d_max_week = math.floor(d_max_week + 0.5)
print(f'item_code: {item_code}')
print(f"D Max / Week : {d_max_week}")

d_max_month = d_max_week * 4


print(f"D Max / Month : {d_max_month}")

# print("Below")
# print(below)

# print("\nAbove")
# print(above)

# print("\nFraction:", fraction)
# print("Below Upper:", below["Upper"])
# print("Above Upper:", above["Upper"])


item_code: 4960.85.150.150
D Max / Week : 38
D Max / Month : 152


In [100]:
safety_stock = round(d_max_month - d_avg_month)

print(f"Safety Stock : {safety_stock}")

Safety Stock : 61


In [101]:
rol = d_avg_month + safety_stock

print(f"ROL : {rol}")

ROL : 152


In [102]:
print("=" * 40)
print(f"Item Code      : {item_code}")
print(f"D Avg / Week   : {d_avg_week:.2f}")
print(f"D Avg / Month  : {d_avg_month}")
print(f"D Max / Week   : {d_max_week}")
print(f"D Max / Month  : {d_max_month}")
print(f"Safety Stock   : {safety_stock}")
print(f"ROL            : {rol}")
print("=" * 40)

Item Code      : 4960.85.150.150
D Avg / Week   : 22.80
D Avg / Month  : 91
D Max / Week   : 38
D Max / Month  : 152
Safety Stock   : 61
ROL            : 152
